[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol2/blob/main/notebooks/ch53_code.ipynb)

# 第53章　ドメイン適応：施設・装置の違いを乗り越える

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 具体的な手法：手間の少ない順に3段階

In [ ]:
import torch

# DANNの要: 勾配反転で「施設を識別できない特徴」を作る
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd): ctx.lambd = lambd; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lambd * g, None   # 勾配を反転

feat = encoder(x)
y_pred = classifier(feat)                    # 病変分類は通常どおり学習する
d_pred = domain_head(GradReverse.apply(feat, lambd))  # 特徴から施設を識別できないようにする
loss = task_loss(y_pred, y) + domain_loss(d_pred, domain_label)

## 推論時に適応する：テスト時適応（TTA/TENT）

In [ ]:
configure_bn_only(model)                 # γ,β だけ requires_grad=True
for batch in test_stream:                # 推論しながら、その場で適応
    logits = model(batch)
    loss = softmax_entropy(logits).mean() # 確信度を高める方向へ
    loss.backward(); optimizer.step(); optimizer.zero_grad()
    preds = logits.argmax(1)   # 予測は更新前の値（適応は次バッチから）